# عملیات مورفولوژیکی / Morphological Operations

**هدف:** یادگیری و پیاده‌سازی عملیات مورفولوژیکی برای پردازش تصاویر دودویی

**Objective:** Learn and implement morphological operations for binary image processing

---

## محتوا / Contents:
1. فرسایش / Erosion
2. اتساع / Dilation
3. باز کردن / Opening
4. بسته کردن / Closing
5. گرادیان مورفولوژیکی / Morphological Gradient
6. Top-hat و Black-hat
7. کاربردهای عملی / Practical Applications

In [ ]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple

# تنظیمات نمایش / Display settings
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. توابع کمکی / Helper Functions

In [ ]:
def display_images(images: list, titles: list, cmap='gray'):
    """
    نمایش چند تصویر در کنار هم
    """
    n = len(images)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if rows > 1 else axes
    
    for i, (img, title) in enumerate(zip(images, titles)):
        axes[i].imshow(img, cmap=cmap)
        axes[i].set_title(title, fontsize=12, fontweight='bold')
        axes[i].axis('off')
    
    # حذف محورهای اضافی
    for i in range(n, len(axes)):
        axes[i].remove()
    
    plt.tight_layout()
    plt.show()


def add_salt_pepper_noise(image: np.ndarray, salt_prob: float = 0.01, 
                          pepper_prob: float = 0.01) -> np.ndarray:
    """
    اضافه کردن نویز نمک و فلفل به تصویر
    
    Args:
        image: تصویر ورودی
        salt_prob: احتمال نویز سفید (نمک)
        pepper_prob: احتمال نویز سیاه (فلفل)
    
    Returns:
        تصویر نویزی
    """
    noisy = image.copy()
    
    # نویز نمک (سفید)
    salt_mask = np.random.random(image.shape) < salt_prob
    noisy[salt_mask] = 255
    
    # نویز فلفل (سیاه)
    pepper_mask = np.random.random(image.shape) < pepper_prob
    noisy[pepper_mask] = 0
    
    return noisy

print("✓ توابع کمکی آماده شدند")

## 2. ایجاد تصویر نمونه / Create Sample Image

In [ ]:
# ایجاد تصویر دودویی با اشکال مختلف
# Create binary image with different shapes
image = np.zeros((400, 600), dtype=np.uint8)

# اضافه کردن مستطیل‌ها
cv2.rectangle(image, (50, 50), (200, 150), 255, -1)
cv2.rectangle(image, (250, 50), (400, 150), 255, -1)

# اضافه کردن دایره‌ها
cv2.circle(image, (125, 250), 60, 255, -1)
cv2.circle(image, (325, 250), 60, 255, -1)

# اضافه کردن خط
cv2.line(image, (100, 320), (500, 320), 255, 10)

# اضافه کردن نویز نمک و فلفل
noisy_image = add_salt_pepper_noise(image, salt_prob=0.02, pepper_prob=0.02)

display_images([image, noisy_image], 
               ['تصویر اصلی / Original', 'تصویر نویزی / Noisy'])

print(f"اندازه تصویر / Image size: {image.shape}")

## 3. عنصر ساختاری / Structuring Element

عنصر ساختاری (Kernel) شکل و اندازه همسایگی را که در عملیات مورفولوژیکی استفاده می‌شود، تعیین می‌کند.

The structuring element (kernel) defines the shape and size of the neighborhood used in morphological operations.

In [ ]:
# ایجاد عناصر ساختاری مختلف
# Create different structuring elements
kernel_rect = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
kernel_ellipse = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
kernel_cross = cv2.getStructuringElement(cv2.MORPH_CROSS, (5, 5))

# نمایش عناصر ساختاری
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

kernels = [kernel_rect, kernel_ellipse, kernel_cross]
titles = ['مستطیل / Rectangle', 'بیضی / Ellipse', 'صلیب / Cross']

for ax, kernel, title in zip(axes, kernels, titles):
    ax.imshow(kernel, cmap='gray', interpolation='nearest')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
    # نمایش مقادیر
    for i in range(kernel.shape[0]):
        for j in range(kernel.shape[1]):
            ax.text(j, i, str(kernel[i, j]), ha='center', va='center', color='red')

plt.tight_layout()
plt.show()

print("✓ عناصر ساختاری ایجاد شدند")

## 4. فرسایش / Erosion

فرسایش باعث کوچک شدن اشیاء سفید و حذف نویزهای کوچک می‌شود.

Erosion shrinks white objects and removes small noise.

In [ ]:
# اعمال فرسایش با اندازه‌های مختلف
# Apply erosion with different sizes
kernel_3 = np.ones((3, 3), np.uint8)
kernel_5 = np.ones((5, 5), np.uint8)
kernel_7 = np.ones((7, 7), np.uint8)

eroded_3 = cv2.erode(noisy_image, kernel_3, iterations=1)
eroded_5 = cv2.erode(noisy_image, kernel_5, iterations=1)
eroded_7 = cv2.erode(noisy_image, kernel_7, iterations=1)

# نمایش نتایج
images = [noisy_image, eroded_3, eroded_5, eroded_7]
titles = ['اصلی / Original', 
          'فرسایش 3×3 / Erosion 3×3',
          'فرسایش 5×5 / Erosion 5×5',
          'فرسایش 7×7 / Erosion 7×7']

display_images(images, titles)

print("✓ فرسایش انجام شد")

## 5. اتساع / Dilation

اتساع باعث بزرگ شدن اشیاء سفید و پر شدن شکاف‌های کوچک می‌شود.

Dilation enlarges white objects and fills small gaps.

In [ ]:
# اعمال اتساع با اندازه‌های مختلف
# Apply dilation with different sizes
dilated_3 = cv2.dilate(noisy_image, kernel_3, iterations=1)
dilated_5 = cv2.dilate(noisy_image, kernel_5, iterations=1)
dilated_7 = cv2.dilate(noisy_image, kernel_7, iterations=1)

# نمایش نتایج
images = [noisy_image, dilated_3, dilated_5, dilated_7]
titles = ['اصلی / Original', 
          'اتساع 3×3 / Dilation 3×3',
          'اتساع 5×5 / Dilation 5×5',
          'اتساع 7×7 / Dilation 7×7']

display_images(images, titles)

print("✓ اتساع انجام شد")

## 6. باز کردن / Opening

باز کردن = فرسایش + اتساع

برای حذف نویزهای کوچک استفاده می‌شود.

Opening = Erosion + Dilation

Used to remove small noise.

In [ ]:
# اعمال باز کردن
# Apply opening
opening = cv2.morphologyEx(noisy_image, cv2.MORPH_OPEN, kernel_5)

# مقایسه با فرسایش و اتساع جداگانه
# Compare with separate erosion and dilation
eroded = cv2.erode(noisy_image, kernel_5, iterations=1)
eroded_dilated = cv2.dilate(eroded, kernel_5, iterations=1)

# نمایش نتایج
images = [noisy_image, eroded, eroded_dilated, opening]
titles = ['اصلی / Original',
          'فرسایش / Erosion',
          'فرسایش + اتساع / Erosion + Dilation',
          'باز کردن / Opening']

display_images(images, titles)

print("✓ باز کردن انجام شد")

## 7. بسته کردن / Closing

بسته کردن = اتساع + فرسایش

برای پر کردن شکاف‌های کوچک استفاده می‌شود.

Closing = Dilation + Erosion

Used to fill small gaps.

In [ ]:
# ایجاد تصویر با شکاف
# Create image with gaps
image_with_gaps = np.zeros((300, 400), dtype=np.uint8)
cv2.rectangle(image_with_gaps, (50, 50), (350, 250), 255, 20)

# اضافه کردن شکاف‌ها
cv2.rectangle(image_with_gaps, (100, 45), (110, 55), 0, -1)
cv2.rectangle(image_with_gaps, (200, 45), (210, 55), 0, -1)
cv2.rectangle(image_with_gaps, (300, 45), (310, 55), 0, -1)

# اعمال بسته کردن
# Apply closing
closing = cv2.morphologyEx(image_with_gaps, cv2.MORPH_CLOSE, kernel_7)

# مقایسه با اتساع و فرسایش جداگانه
dilated = cv2.dilate(image_with_gaps, kernel_7, iterations=1)
dilated_eroded = cv2.erode(dilated, kernel_7, iterations=1)

# نمایش نتایج
images = [image_with_gaps, dilated, dilated_eroded, closing]
titles = ['اصلی (با شکاف) / Original (with gaps)',
          'اتساع / Dilation',
          'اتساع + فرسایش / Dilation + Erosion',
          'بسته کردن / Closing']

display_images(images, titles)

print("✓ بسته کردن انجام شد")

## 8. گرادیان مورفولوژیکی / Morphological Gradient

گرادیان = اتساع - فرسایش

برای استخراج مرزهای اشیاء استفاده می‌شود.

Gradient = Dilation - Erosion

Used to extract object boundaries.

In [ ]:
# اعمال گرادیان مورفولوژیکی
# Apply morphological gradient
gradient = cv2.morphologyEx(image, cv2.MORPH_GRADIENT, kernel_5)

# محاسبه دستی
dilated = cv2.dilate(image, kernel_5, iterations=1)
eroded = cv2.erode(image, kernel_5, iterations=1)
gradient_manual = dilated - eroded

# نمایش نتایج
images = [image, dilated, eroded, gradient, gradient_manual]
titles = ['اصلی / Original',
          'اتساع / Dilation',
          'فرسایش / Erosion',
          'گرادیان / Gradient',
          'گرادیان دستی / Manual Gradient']

display_images(images, titles)

print("✓ گرادیان مورفولوژیکی محاسبه شد")

## 9. Top-hat و Black-hat

**Top-hat:** تصویر اصلی - باز کردن (استخراج نقاط روشن کوچک)

**Black-hat:** بسته کردن - تصویر اصلی (استخراج نقاط تیره کوچک)

**Top-hat:** Original - Opening (extract small bright spots)

**Black-hat:** Closing - Original (extract small dark spots)

In [ ]:
# ایجاد تصویر با نقاط روشن و تیره کوچک
# Create image with small bright and dark spots
test_image = np.ones((300, 400), dtype=np.uint8) * 128

# اضافه کردن اشکال بزرگ
cv2.rectangle(test_image, (50, 50), (150, 150), 200, -1)
cv2.circle(test_image, (300, 100), 50, 60, -1)

# اضافه کردن نقاط کوچک روشن
for i in range(10):
    x, y = np.random.randint(50, 350), np.random.randint(50, 250)
    cv2.circle(test_image, (x, y), 3, 255, -1)

# اضافه کردن نقاط کوچک تیره
for i in range(10):
    x, y = np.random.randint(50, 350), np.random.randint(50, 250)
    cv2.circle(test_image, (x, y), 3, 0, -1)

# اعمال Top-hat و Black-hat
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
tophat = cv2.morphologyEx(test_image, cv2.MORPH_TOPHAT, kernel)
blackhat = cv2.morphologyEx(test_image, cv2.MORPH_BLACKHAT, kernel)

# نمایش نتایج
images = [test_image, tophat, blackhat]
titles = ['اصلی / Original',
          'Top-hat (نقاط روشن / Bright spots)',
          'Black-hat (نقاط تیره / Dark spots)']

display_images(images, titles)

print("✓ Top-hat و Black-hat محاسبه شد")

## 10. کاربردهای عملی / Practical Applications

### کاربرد 1: حذف نویز / Noise Removal

In [ ]:
# حذف نویز با باز کردن
# Remove noise with opening
kernel = np.ones((3, 3), np.uint8)
denoised = cv2.morphologyEx(noisy_image, cv2.MORPH_OPEN, kernel, iterations=2)

# نمایش نتایج
images = [noisy_image, denoised]
titles = ['نویزی / Noisy', 'بدون نویز / Denoised']

display_images(images, titles)

print("✓ نویز حذف شد")

### کاربرد 2: پر کردن شکاف‌ها / Gap Filling

In [ ]:
# پر کردن شکاف‌ها با بسته کردن
# Fill gaps with closing
kernel = np.ones((7, 7), np.uint8)
filled = cv2.morphologyEx(image_with_gaps, cv2.MORPH_CLOSE, kernel)

# نمایش نتایج
images = [image_with_gaps, filled]
titles = ['با شکاف / With gaps', 'پر شده / Filled']

display_images(images, titles)

print("✓ شکاف‌ها پر شدند")

### کاربرد 3: استخراج مرزها / Boundary Extraction

In [ ]:
# استخراج مرزها با گرادیان
# Extract boundaries with gradient
kernel = np.ones((3, 3), np.uint8)
boundaries = cv2.morphologyEx(image, cv2.MORPH_GRADIENT, kernel)

# نمایش نتایج
images = [image, boundaries]
titles = ['اصلی / Original', 'مرزها / Boundaries']

display_images(images, titles)

print("✓ مرزها استخراج شدند")

## 11. تمرین‌ها / Exercises

### تمرین 1 / Exercise 1
تصویری با نویز زیاد ایجاد کنید و با استفاده از عملیات مورفولوژیکی آن را تمیز کنید.

Create an image with high noise and clean it using morphological operations.

### تمرین 2 / Exercise 2
اندازه و شکل عنصر ساختاری را تغییر دهید و تأثیر آن را بر نتایج بررسی کنید.

Change the size and shape of the structuring element and examine its effect on the results.

### تمرین 3 / Exercise 3
ترکیبی از عملیات مورفولوژیکی را برای حل یک مسئله خاص طراحی کنید.

Design a combination of morphological operations to solve a specific problem.